# Random Forest Machine Learning Project
## Titanic Passenger Survival Prediction
**Prepared by: KEERTHI H**

This notebook builds a Random Forest classification model using the Kaggle Titanic training dataset. The target is `Survived` (0 = did not survive, 1 = survived).

**Dataset requirement:** the training file contains 891 records, exceeding the 500-row minimum in the project guidelines.

> Run the notebook from top to bottom in Google Colab or Jupyter. It downloads the public CSV mirror of Kaggle's training file. The official Kaggle competition page is cited in the report.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)


## 2. Load the Kaggle dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/train.csv"
df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn information:")
df.info()


## 3. Basic data exploration

In [ ]:
display(df.describe(include="all").T)
print("\\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("\\nTarget distribution:")
display(df["Survived"].value_counts().rename(index={0: "Did not survive", 1: "Survived"}))


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Survived")
plt.title("Passenger Survival Count")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.tight_layout()
plt.show()


## 4. Feature preparation

In [ ]:
data = df.copy()

# Create useful family-related features and extract passenger title.
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
data["Title"] = data["Name"].str.extract(r",\\s*([^.]*)\\.", expand=False).fillna("Unknown")

# Group uncommon titles to reduce sparse categories.
title_counts = data["Title"].value_counts()
data["Title"] = data["Title"].where(data["Title"].map(title_counts) >= 10, "Rare")

features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "FamilySize", "IsAlone", "Title"
]
X = data[features]
y = data["Survived"]

numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"
]
categorical_features = ["Sex", "Embarked", "Title"]

print("Feature matrix:", X.shape)
print("Target:", y.shape)


## 5. Handle missing values and encode categories

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## 6. Train-test split and Random Forest model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
print("Training completed.")


## 7. Make predictions and evaluate

In [ ]:
y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

metrics_df = pd.DataFrame(
    {"Metric": list(results.keys()), "Score": list(results.values())}
)
display(metrics_df.round(4))
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))


## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()
print("Confusion matrix (rows = actual, columns = predicted):\\n", cm)


## 9. Feature importance

In [ ]:
# Impurity-based feature importance from the fitted Random Forest.
fitted_preprocessor = rf_pipeline.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = rf_pipeline.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(12)
)
display(importance_df)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()


## 10. Conclusion

The notebook trains a Random Forest classifier to predict passenger survival. The workflow includes data exploration, missing-value imputation, categorical encoding, feature engineering, train-test splitting, model training, evaluation and feature-importance visualization.

The exact scores are generated when the notebook is executed. The report supplied with this notebook records the results from a stratified 80:20 split using `random_state=42`.

**Limitation:** this is a historical passenger dataset. Model performance on this dataset does not imply performance on modern populations or unrelated prediction tasks.
